# 02 Silver Layer — Data Cleaning & Transformation

The Silver layer cleans and standardizes the raw Bronze data.

Transformations performed:
- Remove duplicates
- Drop null values in critical columns
- Fix column formats (lowercase, trimmed)
- Remove unnecessary columns
- Ensure correct data types

This prepares the dataset for analysis and modeling.

In [0]:
BRONZE_TABLE = "bronze_spotify"
SILVER_TABLE = "silver_spotify"

df = spark.table(BRONZE_TABLE)

display(df)

_c0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.461,1,-6.746,0,0.143,0.0322,1.01E-6,0.358,0.715,87.917,4.0,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.42,0.166,1,-17.235,1,0.0763,0.924,5.56E-6,0.101,0.267,77.489,4.0,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.359,0,-9.734,1,0.0557,0.21,0.0,0.117,0.12,76.332,4.0,acoustic
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Soundtrack),Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.905,7.07E-5,0.132,0.143,181.74,3.0,acoustic
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.443,2,-9.681,1,0.0526,0.469,0.0,0.0829,0.167,119.949,4.0,acoustic
5,01MVOl9KtVTNfFiBU9I7dc,Tyrone Wells,Days I Will Remember,Days I Will Remember,58,214240,False,0.688,0.481,6,-8.807,1,0.105,0.289,0.0,0.189,0.666,98.017,4.0,acoustic
6,6Vc5wAMmXdKIAM7WUoEb7N,A Great Big World;Christina Aguilera,Is There Anybody Out There?,Say Something,74,229400,False,0.407,0.147,2,-8.822,1,0.0355,0.857,2.89E-6,0.0913,0.0765,141.284,3.0,acoustic
7,1EzrEOXmMH3G43AXT1y7pA,Jason Mraz,We Sing. We Dance. We Steal Things.,I'm Yours,80,242946,False,0.703,0.444,11,-9.331,1,0.0417,0.559,0.0,0.0973,0.712,150.96,4.0,acoustic
8,0IktbUcnAGrvD03AWnz3Q8,Jason Mraz;Colbie Caillat,We Sing. We Dance. We Steal Things.,Lucky,74,189613,False,0.625,0.414,0,-8.7,1,0.0369,0.294,0.0,0.151,0.669,130.088,4.0,acoustic
9,7k9GuJYLp2AzqokyEdwEw2,Ross Copperman,Hunger,Hunger,56,205594,False,0.442,0.632,1,-6.77,1,0.0295,0.426,0.00419,0.0735,0.196,78.899,4.0,acoustic


## Initial Data Inspection

In [0]:
print("Row count before cleaning:", df.count())
print("Columns:", df.columns)

df.printSchema()

Row count before cleaning: 114000
Columns: ['_c0', 'track_id', 'artists', 'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre']
root
 |-- _c0: integer (nullable = true)
 |-- track_id: string (nullable = true)
 |-- artists: string (nullable = true)
 |-- album_name: string (nullable = true)
 |-- track_name: string (nullable = true)
 |-- popularity: string (nullable = true)
 |-- duration_ms: string (nullable = true)
 |-- explicit: string (nullable = true)
 |-- danceability: string (nullable = true)
 |-- energy: string (nullable = true)
 |-- key: string (nullable = true)
 |-- loudness: string (nullable = true)
 |-- mode: string (nullable = true)
 |-- speechiness: string (nullable = true)
 |-- acousticness: string (nullable = true)
 |-- instrumentalness: double (nullable = true)
 |-- liveness: string (nullable

## Data Cleaning Steps

In [0]:
from pyspark.sql.functions import col, trim, lower

# Remove unwanted index column if present
if "_c0" in df.columns:
    df = df.drop("_c0")

# Drop duplicates
df = df.dropDuplicates()

# Drop rows with nulls in important columns
df = df.dropna(subset=["track_name", "artists", "popularity"])

# Clean text columns
df = (
    df
    .withColumn("track_name", trim(col("track_name")))
    .withColumn("artists", trim(col("artists")))
    .withColumn("album_name", trim(col("album_name")))
    .withColumn("track_genre", lower(trim(col("track_genre"))))
)

## Data Type Fixing

In [0]:
from pyspark.sql.functions import expr

numeric_cols = [
    "popularity", "duration_ms", "danceability", "energy",
    "loudness", "speechiness", "acousticness",
    "instrumentalness", "liveness", "valence", "tempo"
]

for c in numeric_cols:
    if c in df.columns:
        df = df.withColumn(c, expr(f"try_cast(`{c}` as double)"))

In [0]:
df = df.dropna(subset=[
    "popularity", "danceability", "energy", "valence",
    "acousticness", "instrumentalness", "tempo",
    "loudness", "speechiness"
])

## Save Silver Table

In [0]:
from pyspark.sql.functions import expr

# Re-read Bronze fresh
df = spark.table("bronze_spotify")

# Drop index column
if "_c0" in df.columns:
    df = df.drop("_c0")

# Drop duplicates
df = df.dropDuplicates()

# Text cleaning
from pyspark.sql.functions import col, trim, lower

df = (
    df
    .withColumn("track_name", trim(col("track_name")))
    .withColumn("artists", trim(col("artists")))
    .withColumn("album_name", trim(col("album_name")))
    .withColumn("track_genre", lower(trim(col("track_genre"))))
)

# Safe numeric casting
numeric_cols = [
    "popularity", "duration_ms", "danceability", "energy",
    "loudness", "speechiness", "acousticness",
    "instrumentalness", "liveness", "valence", "tempo"
]

for c in numeric_cols:
    df = df.withColumn(c, expr(f"try_cast(`{c}` as double)"))

# Remove bad rows after safe cast
df = df.dropna(subset=[
    "popularity", "danceability", "energy", "valence",
    "acousticness", "instrumentalness", "tempo",
    "loudness", "speechiness"
])

# Save Silver
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver_spotify")

print("Silver table created: silver_spotify")
print("Rows:", df.count())
display(df.limit(10))

Silver table created: silver_spotify
Rows: 113421


track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73.0,230666.0,False,0.676,0.461,1,-6.746,0,0.143,0.0322,1.01E-6,0.358,0.715,87.917,4.0,acoustic
4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55.0,149610.0,False,0.42,0.166,1,-17.235,1,0.0763,0.924,5.56E-6,0.101,0.267,77.489,4.0,acoustic
1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57.0,210826.0,False,0.438,0.359,0,-9.734,1,0.0557,0.21,0.0,0.117,0.12,76.332,4.0,acoustic
6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Soundtrack),Can't Help Falling In Love,71.0,201933.0,False,0.266,0.0596,0,-18.515,1,0.0363,0.905,7.07E-5,0.132,0.143,181.74,3.0,acoustic
5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82.0,198853.0,False,0.618,0.443,2,-9.681,1,0.0526,0.469,0.0,0.0829,0.167,119.949,4.0,acoustic
01MVOl9KtVTNfFiBU9I7dc,Tyrone Wells,Days I Will Remember,Days I Will Remember,58.0,214240.0,False,0.688,0.481,6,-8.807,1,0.105,0.289,0.0,0.189,0.666,98.017,4.0,acoustic
6Vc5wAMmXdKIAM7WUoEb7N,A Great Big World;Christina Aguilera,Is There Anybody Out There?,Say Something,74.0,229400.0,False,0.407,0.147,2,-8.822,1,0.0355,0.857,2.89E-6,0.0913,0.0765,141.284,3.0,acoustic
1EzrEOXmMH3G43AXT1y7pA,Jason Mraz,We Sing. We Dance. We Steal Things.,I'm Yours,80.0,242946.0,False,0.703,0.444,11,-9.331,1,0.0417,0.559,0.0,0.0973,0.712,150.96,4.0,acoustic
0IktbUcnAGrvD03AWnz3Q8,Jason Mraz;Colbie Caillat,We Sing. We Dance. We Steal Things.,Lucky,74.0,189613.0,False,0.625,0.414,0,-8.7,1,0.0369,0.294,0.0,0.151,0.669,130.088,4.0,acoustic
7k9GuJYLp2AzqokyEdwEw2,Ross Copperman,Hunger,Hunger,56.0,205594.0,False,0.442,0.632,1,-6.77,1,0.0295,0.426,0.00419,0.0735,0.196,78.899,4.0,acoustic


In [0]:
spark.sql("SHOW TABLES").show()

+--------+--------------+-----------+
|database|     tableName|isTemporary|
+--------+--------------+-----------+
| default|bronze_spotify|      false|
| default|silver_spotify|      false|
+--------+--------------+-----------+

